# 05. Análisis económico y diseño de planes

Este notebook analiza la sostenibilidad económica del modelo freemium de PROXUS, combinando datos calculados a partir de las tablas disponibles con información agregada proporcionada por la empresa. El objetivo es estimar ingresos, costes, márgenes, KPIs de negocio y escenarios de sostenibilidad para apoyar recomendaciones sobre pricing, retención y diseño de planes.

## Estructura del notebook

1. Preparación de datos económicos y segmentos de usuarios.
2. Cálculo de ingresos Stripe de enero de 2026.
3. Estimación y conciliación de costes IA.
4. Incorporación de costes operativos, fijos y marketing proporcionados por PROXUS.
5. Validación de ingresos, comisiones y resultado económico.
6. Cálculo de KPIs de negocio.
7. Punto muerto y escenarios de sostenibilidad.
8. Análisis por plan y recomendaciones.

Para el análisis económico se aplica el criterio de utilizar métricas calculadas directamente a partir de las tablas siempre que sea posible. Por ello, los ingresos se calculan desde `stripe_charges`, el coste IA desde `ai_spans.cost` y las comisiones Stripe mediante la tarifa aplicable al volumen de cobros de enero. Las partidas no observables directamente en la base de datos, como infraestructura, costes fijos y marketing, se incorporan a partir de la información proporcionada por PROXUS.

En concreto, el coste de infraestructura, servidor, base de datos y compilaciones se fija en 1.100 € para enero de 2026. Esta partida se trata como coste fijo operativo de corto plazo. El coste variable unitario del punto muerto incluye únicamente IA calculada y comisiones de cobro.

In [81]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_PATH = Path("../data")
OUTPUT_PATH = Path("../outputs")

OUTPUT_PATH.mkdir(exist_ok=True)

In [82]:
def parse_dates(series):
    return pd.to_datetime(series.astype(str).str.strip('"'), errors="coerce", utc=True)


def clean_id(col):
    """
    Normaliza identificadores como string.
    """
    return col.astype(str).str.strip()


def clean_amount_eur(col):
    """
    Convierte importes a numérico.
    Si los importes vienen en céntimos, habrá que dividir entre 100 posteriormente.
    """
    return pd.to_numeric(col, errors="coerce")


def resumen_importe(nombre, valor):
    print(f"{nombre}: {valor:,.2f} €")

In [83]:
users = pd.read_csv(DATA_PATH / "users.csv", low_memory=False)

stripe_charges = pd.read_csv(DATA_PATH / "stripe_charges.csv", low_memory=False)
stripe_subscriptions = pd.read_csv(DATA_PATH / "stripe_subscriptions.csv", low_memory=False)

# Estas dos pueden existir o no según tu exportación.
# Si alguna da error, la comentamos.
try:
    stripe_invoice_line_items = pd.read_csv(DATA_PATH / "stripe_invoice_line_items.csv", low_memory=False)
    print("stripe_invoice_line_items cargada")
except FileNotFoundError:
    stripe_invoice_line_items = None
    print("stripe_invoice_line_items no encontrada")

try:
    stripe_coupons = pd.read_csv(DATA_PATH / "stripe_coupons.csv", low_memory=False)
    print("stripe_coupons cargada")
except FileNotFoundError:
    stripe_coupons = None
    print("stripe_coupons no encontrada")

traces = pd.read_csv(DATA_PATH / "traces.csv", low_memory=False, sep=";")
ai_spans = pd.read_csv(DATA_PATH / "ai_spans.csv", low_memory=False)

print("users:", users.shape)
print("stripe_charges:", stripe_charges.shape)
print("stripe_subscriptions:", stripe_subscriptions.shape)
print("traces:", traces.shape)
print("ai_spans:", ai_spans.shape)

stripe_invoice_line_items cargada
stripe_coupons cargada
users: (32636, 32)
stripe_charges: (703, 25)
stripe_subscriptions: (627, 32)
traces: (112353, 6)
ai_spans: (269383, 16)


In [84]:
users["id"] = clean_id(users["id"])
users["created_at"] = parse_dates(users["created_at"])
users["subscription_created_at"] = parse_dates(users["subscription_created_at"])

users["subscription_type_clean"] = (
    users["subscription_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

users["stripe_customer_id_clean"] = (
    users["stripe_customer_id"]
    .astype(str)
    .str.strip()
    .replace({
        "": np.nan,
        "nan": np.nan,
        "none": np.nan,
        "None": np.nan,
        "NaN": np.nan,
        "NULL": np.nan,
        "null": np.nan
    })
)

users["es_free_puro"] = (
    (users["subscription_type_clean"] == "free") &
    (users["stripe_customer_id_clean"].isna())
)

users["ha_pagado_alguna_vez"] = users["stripe_customer_id_clean"].notna()

users["es_pagador_activo"] = (
    users["stripe_customer_id_clean"].notna() &
    (users["subscription_type_clean"] != "free")
)

users["es_churned"] = (
    users["stripe_customer_id_clean"].notna() &
    (users["subscription_type_clean"] == "free")
)

print("Usuarios totales:", len(users))
print("Free puros:", users["es_free_puro"].sum())
print("Pagadores activos:", users["es_pagador_activo"].sum())
print("Churned:", users["es_churned"].sum())
print("Alguna vez pagaron:", users["ha_pagado_alguna_vez"].sum())

Usuarios totales: 32636
Free puros: 32017
Pagadores activos: 516
Churned: 103
Alguna vez pagaron: 619


### Inspección inicial de la tabla de cobros Stripe

Se revisan las columnas y valores de `stripe_charges` para confirmar qué campo contiene el importe, la fecha y el estado del pago.

In [85]:
stripe_charges.head()

,id,customer,amount,amount_captured,amount_refunded,currency,status,paid,refunded,captured,disputed,created,invoice,payment_intent,payment_method,payment_method_type,card_brand,card_funding,outcome_type,outcome_reason,outcome_risk_level,outcome_risk_score,failure_code,failure_message,metadata
0,ch_3SwqxLQNcL1q3Wv80dkWm5r5,cus_Tj3LKRdWTPnSXl,699,699,0,eur,succeeded,True,False,True,False,2026-02-07T22:00:59.000Z,NaN,pi_3SwqxLQNcL1q3Wv80kHu8uDD,pm_1SlbEjQNcL1q3Wv8Be4oTlOq,card,mastercard,debit,authorized,NaN,normal,NaN,NaN,NaN,NaN
1,ch_3SyIwGQNcL1q3Wv81YiXTyok,cus_TkYJ8qu1k0PPB7,699,699,0,eur,succeeded,True,False,True,False,2026-02-07T21:21:14.000Z,NaN,pi_3SyIwGQNcL1q3Wv81R0WMmF7,pm_1Sn3DAQNcL1q3Wv83FZHVIaU,card,visa,debit,authorized,NaN,normal,NaN,NaN,NaN,NaN
2,ch_3SyIpBQNcL1q3Wv80XPwwrCx,cus_TkYA3etqqoixxU,699,699,0,eur,succeeded,True,False,True,False,2026-02-07T21:13:55.000Z,NaN,pi_3SyIpBQNcL1q3Wv805VXqWNp,pm_1Sn363QNcL1q3Wv8O8WSwLnh,card,visa,debit,authorized,NaN,normal,NaN,NaN,NaN,NaN
3,ch_3SyHngQNcL1q3Wv80OUgjjUP,cus_TkX9ED1gFUL4Qt,699,699,0,eur,succeeded,True,False,True,False,2026-02-07T20:08:18.000Z,NaN,pi_3SyHngQNcL1q3Wv80zUTvqd0,pm_1Sn24zQNcL1q3Wv8xDxPV7xv,card,visa,debit,authorized,NaN,normal,NaN,NaN,NaN,NaN
4,ch_3SyHhJQNcL1q3Wv803fkfagO,cus_TkX1fyfw9lkkD9,699,699,0,eur,succeeded,True,False,True,False,2026-02-07T20:01:43.000Z,NaN,pi_3SyHhJQNcL1q3Wv80EptgrGW,pm_1Sn1xlQNcL1q3Wv8YGhytEC1,card,mastercard,debit,authorized,NaN,normal,NaN,NaN,NaN,NaN


In [86]:
stripe_charges.columns.tolist()

['id',
 'customer',
 'amount',
 'amount_captured',
 'amount_refunded',
 'currency',
 'status',
 'paid',
 'refunded',
 'captured',
 'disputed',
 'created',
 'invoice',
 'payment_intent',
 'payment_method',
 'payment_method_type',
 'card_brand',
 'card_funding',
 'outcome_type',
 'outcome_reason',
 'outcome_risk_level',
 'outcome_risk_score',
 'failure_code',
 'failure_message',
 'metadata']

In [87]:
charges = stripe_charges.copy()

# Normalizar fecha
if "created" in charges.columns:
    charges["created_at"] = parse_dates(charges["created"])
elif "created_at" in charges.columns:
    charges["created_at"] = parse_dates(charges["created_at"])
else:
    raise ValueError("No encuentro columna de fecha en stripe_charges")

# Normalizar importe
# Stripe suele exportar amount en céntimos, pero en tus notebooks anteriores parecía estar ya tratado.
# Primero inspeccionamos magnitudes.
charges["amount_num"] = pd.to_numeric(charges["amount"], errors="coerce")

print(charges["amount_num"].describe())

count     703.00
mean      814.37
std       394.08
min       349.00
25%       699.00
50%       699.00
75%       699.00
max     3,199.00
Name: amount_num, dtype: float64


In [88]:
# Detectar si amount parece estar en céntimos
mediana_amount = charges["amount_num"].median()

if mediana_amount > 100:
    charges["amount_eur"] = charges["amount_num"] / 100
    print("Importes interpretados como céntimos y convertidos a euros.")
else:
    charges["amount_eur"] = charges["amount_num"]
    print("Importes interpretados como euros.")

# Filtrar cobros exitosos
if "paid" in charges.columns:
    charges["paid_bool"] = charges["paid"].astype(str).str.lower().isin(["true", "1", "yes"])
else:
    charges["paid_bool"] = True

if "status" in charges.columns:
    charges["status_clean"] = charges["status"].astype(str).str.lower()
else:
    charges["status_clean"] = "succeeded"

if "refunded" in charges.columns:
    charges["refunded_bool"] = charges["refunded"].astype(str).str.lower().isin(["true", "1", "yes"])
else:
    charges["refunded_bool"] = False

charges_success = charges[
    (charges["paid_bool"]) &
    (charges["status_clean"].isin(["succeeded", "paid", "captured"])) &
    (~charges["refunded_bool"])
].copy()

print("Cobros totales:", len(charges))
print("Cobros exitosos no reembolsados:", len(charges_success))
print("Ingresos totales exitosos:", charges_success["amount_eur"].sum())

Importes interpretados como céntimos y convertidos a euros.
Cobros totales: 703
Cobros exitosos no reembolsados: 615
Ingresos totales exitosos: 4989.870000000001


In [89]:
inicio_enero = pd.Timestamp("2026-01-01", tz="UTC")
fin_enero = pd.Timestamp("2026-02-01", tz="UTC")

charges_enero = charges_success[
    (charges_success["created_at"] >= inicio_enero) &
    (charges_success["created_at"] < fin_enero)
].copy()

ingresos_brutos_enero = charges_enero["amount_eur"].sum()
n_cobros_enero = len(charges_enero)

# Clientes únicos si existe customer
if "customer" in charges_enero.columns:
    clientes_pago_enero = charges_enero["customer"].nunique()
elif "customer_id" in charges_enero.columns:
    clientes_pago_enero = charges_enero["customer_id"].nunique()
else:
    clientes_pago_enero = np.nan

print("=== INGRESOS STRIPE ENERO 2026 ===")
print(f"Cobros exitosos enero: {n_cobros_enero:,}")
print(f"Clientes únicos con cobro enero: {clientes_pago_enero:,}")
print(f"Ingresos brutos enero: {ingresos_brutos_enero:,.2f} €")

=== INGRESOS STRIPE ENERO 2026 ===
Cobros exitosos enero: 490
Clientes únicos con cobro enero: 489
Ingresos brutos enero: 3,942.99 €


In [90]:
IVA = 0.21

ingresos_netos_sin_iva_enero = ingresos_brutos_enero / (1 + IVA)

print("=== INGRESOS NETOS ESTIMADOS ===")
print(f"Ingresos brutos enero: {ingresos_brutos_enero:,.2f} €")
print(f"Ingresos netos estimados sin IVA: {ingresos_netos_sin_iva_enero:,.2f} €")

=== INGRESOS NETOS ESTIMADOS ===
Ingresos brutos enero: 3,942.99 €
Ingresos netos estimados sin IVA: 3,258.67 €


Se asume que los importes de Stripe corresponden a importes brutos cobrados al usuario final e incluyen IVA. Por tanto, se estima el ingreso neto sin IVA dividiendo los ingresos brutos entre 1,21. Si los importes exportados por Stripe ya estuvieran netos de IVA, esta estimación sería conservadora.

In [91]:
spans = ai_spans.copy()

# Fecha del span
if "created_at" in spans.columns:
    spans["created_at"] = parse_dates(spans["created_at"])
elif "start_time" in spans.columns:
    spans["created_at"] = parse_dates(spans["start_time"])
else:
    raise ValueError("No encuentro columna de fecha en ai_spans")

spans["cost"] = pd.to_numeric(spans["cost"], errors="coerce").fillna(0)

spans_enero = spans[
    (spans["created_at"] >= inicio_enero) &
    (spans["created_at"] < fin_enero)
].copy()

coste_ia_trazado_enero = spans_enero["cost"].sum()

print("=== COSTE IA TRAZADO ENERO 2026 ===")
print(f"Spans enero: {len(spans_enero):,}")
print(f"Coste IA trazado enero: {coste_ia_trazado_enero:,.2f} $")

=== COSTE IA TRAZADO ENERO 2026 ===
Spans enero: 232,551
Coste IA trazado enero: 525.06 $


In [92]:
# ============================================================
# Tipo de cambio USD/EUR enero 2026
# Fuente: Expansión
# ============================================================

tipo_cambio_enero = pd.DataFrame([
    {"fecha": "2026-01-30", "usd_eur": 0.8414},
    {"fecha": "2026-01-29", "usd_eur": 0.8373},
    {"fecha": "2026-01-28", "usd_eur": 0.8375},
    {"fecha": "2026-01-27", "usd_eur": 0.8347},
    {"fecha": "2026-01-26", "usd_eur": 0.8413},
    {"fecha": "2026-01-23", "usd_eur": 0.8495},
    {"fecha": "2026-01-22", "usd_eur": 0.8519},
    {"fecha": "2026-01-21", "usd_eur": 0.8543},
    {"fecha": "2026-01-20", "usd_eur": 0.8519},
    {"fecha": "2026-01-16", "usd_eur": 0.8608},
    {"fecha": "2026-01-15", "usd_eur": 0.8603},
    {"fecha": "2026-01-14", "usd_eur": 0.8583},
    {"fecha": "2026-01-13", "usd_eur": 0.8581},
    {"fecha": "2026-01-12", "usd_eur": 0.8553},
    {"fecha": "2026-01-09", "usd_eur": 0.8590},
    {"fecha": "2026-01-08", "usd_eur": 0.8565},
    {"fecha": "2026-01-07", "usd_eur": 0.8559},
    {"fecha": "2026-01-06", "usd_eur": 0.8542},
    {"fecha": "2026-01-05", "usd_eur": 0.8573},
    {"fecha": "2026-01-02", "usd_eur": 0.8532},
])

tipo_cambio_enero["fecha"] = pd.to_datetime(tipo_cambio_enero["fecha"])

tipo_cambio_medio_usd_eur = tipo_cambio_enero["usd_eur"].mean()

print(f"Tipo de cambio medio USD/EUR enero 2026: {tipo_cambio_medio_usd_eur:.4f}")

Tipo de cambio medio USD/EUR enero 2026: 0.8514


In [93]:
# ============================================================
# Conversión del coste IA trazado de USD a EUR
# ============================================================

# Tipo de cambio medio USD/EUR enero 2026 calculado a partir de Expansión
USD_EUR_ENERO_2026 = tipo_cambio_medio_usd_eur

coste_ia_trazado_usd_enero = coste_ia_trazado_enero
coste_ia_trazado_eur_enero = coste_ia_trazado_usd_enero * USD_EUR_ENERO_2026

conciliacion_ia = pd.DataFrame([
    {
        "concepto": "Coste IA trazado original",
        "importe": coste_ia_trazado_usd_enero,
        "moneda": "USD",
        "fuente": "ai_spans.cost"
    },
    {
        "concepto": "Coste IA trazado convertido a euros",
        "importe": coste_ia_trazado_eur_enero,
        "moneda": "EUR",
        "fuente": "ai_spans.cost × tipo medio USD/EUR enero 2026"
    }
])

display(conciliacion_ia)

print(f"Tipo de cambio medio USD/EUR enero 2026: {USD_EUR_ENERO_2026:.4f}")
print(f"Coste IA trazado original: {coste_ia_trazado_usd_enero:,.2f} $")
print(f"Coste IA trazado convertido: {coste_ia_trazado_eur_enero:,.2f} €")

,concepto,importe,moneda,fuente
0,Coste IA trazado original,525.06,USD,ai_spans.cost
1,Coste IA trazado convertido a euros,447.05,EUR,ai_spans.cost × tipo medio USD/EUR enero 2026


Tipo de cambio medio USD/EUR enero 2026: 0.8514
Coste IA trazado original: 525.06 $
Coste IA trazado convertido: 447.05 €


Se interpreta el campo `ai_spans.cost` como coste trazado en dólares, dado que los proveedores de modelos de IA suelen facturar en USD. Para compararlo con los costes contables proporcionados por PROXUS, se convierte a euros utilizando el tipo de cambio medio USD/EUR de enero de 2026 calculado a partir de los datos diarios publicados por Expansión.

In [94]:
comision_pct = 0.022
comision_fija = 0.25

comisiones_stripe_estimadas = ingresos_brutos_enero * comision_pct + n_cobros_enero * comision_fija

print("=== COMISIONES STRIPE ENERO ===")
print(f"Comisiones Stripe estimadas: {comisiones_stripe_estimadas:,.2f} €")

=== COMISIONES STRIPE ENERO ===
Comisiones Stripe estimadas: 209.25 €


In [95]:
# ============================================================
# Costes de enero: calculados vs proporcionados por PROXUS
# ============================================================

# Coste IA calculado desde ai_spans y convertido a euros
coste_ia_analisis = coste_ia_trazado_eur_enero

# Coste de infraestructura, servidor, BBDD y compilaciones.
# Partida no observable directamente en las tablas.
coste_infraestructura_empresa = 1100.00

# Coste operativo de plataforma usado en el análisis
coste_plataforma_analisis = (
    coste_ia_analisis
    + coste_infraestructura_empresa
)

# Costes fijos proporcionados por PROXUS
costes_fijos_empresa = 6000.00
coste_sueldos = 4000.00
coste_oficina_gastos = 2000.00

# Marketing enero proporcionado por PROXUS
marketing_enero = 12000.00

costes_enero = pd.DataFrame([
    {
        "partida": "IA trazada",
        "importe": coste_ia_analisis,
        "bloque": "Coste variable",
        "fuente": "Calculado desde ai_spans y convertido a EUR"
    },
    {
        "partida": "Infraestructura, servidor, BBDD y compilaciones",
        "importe": coste_infraestructura_empresa,
        "bloque": "Coste operativo plataforma",
        "fuente": "Estimación a partir de información proporcionada por PROXUS"
    },
    {
        "partida": "Comisiones Stripe",
        "importe": comisiones_stripe_estimadas,
        "bloque": "Coste variable de cobro",
        "fuente": "Estimado a partir de tarifa Stripe"
    },
    {
        "partida": "Sueldos",
        "importe": coste_sueldos,
        "bloque": "Coste fijo estructura",
        "fuente": "PROXUS"
    },
    {
        "partida": "Oficina y gastos menores",
        "importe": coste_oficina_gastos,
        "bloque": "Coste fijo estructura",
        "fuente": "PROXUS"
    },
    {
        "partida": "Marketing creadores TikTok",
        "importe": marketing_enero,
        "bloque": "Marketing",
        "fuente": "PROXUS"
    }
])

costes_enero

,partida,importe,bloque,fuente
0,IA trazada,447.05,Coste variable,Calculado desde ai_spans y convertido a EUR
1,"Infraestructura, servidor, BBDD y compilaciones","1,100.00",Coste operativo plataforma,Estimación a partir de información proporciona...
2,Comisiones Stripe,209.25,Coste variable de cobro,Estimado a partir de tarifa Stripe
3,Sueldos,"4,000.00",Coste fijo estructura,PROXUS
4,Oficina y gastos menores,"2,000.00",Coste fijo estructura,PROXUS
5,Marketing creadores TikTok,"12,000.00",Marketing,PROXUS


In [96]:
# ============================================================
# Márgenes de enero usando coste IA calculado
# ============================================================

margen_ia_trazado = (
    ingresos_netos_sin_iva_enero
    - coste_ia_analisis
)

margen_operativo_plataforma = (
    ingresos_netos_sin_iva_enero
    - coste_plataforma_analisis
    - comisiones_stripe_estimadas
)

resultado_enero_completo = (
    ingresos_netos_sin_iva_enero
    - coste_plataforma_analisis
    - comisiones_stripe_estimadas
    - costes_fijos_empresa
    - marketing_enero
)

resumen_margenes = pd.DataFrame([
    {
        "nivel": "Margen frente a IA trazada",
        "formula": "Ingresos netos - IA calculada",
        "resultado": margen_ia_trazado
    },
    {
        "nivel": "Margen operativo plataforma",
        "formula": "Ingresos netos - IA calculada - infraestructura - Stripe",
        "resultado": margen_operativo_plataforma
    },
    {
        "nivel": "Resultado enero completo",
        "formula": "Ingresos netos - IA calculada - infraestructura - Stripe - fijos - marketing",
        "resultado": resultado_enero_completo
    }
])

resumen_margenes.round(2)

,nivel,formula,resultado
0,Margen frente a IA trazada,Ingresos netos - IA calculada,"2,811.62"
1,Margen operativo plataforma,Ingresos netos - IA calculada - infraestructur...,"1,502.37"
2,Resultado enero completo,Ingresos netos - IA calculada - infraestructur...,"-16,497.63"


In [122]:
# ============================================================
# Resumen económico enero 2026
# ============================================================

print("=== RESUMEN ECONÓMICO ENERO 2026 ===")
resumen_importe("Ingresos brutos Stripe", ingresos_brutos_enero)
resumen_importe("Ingresos netos estimados sin IVA", ingresos_netos_sin_iva_enero)

print()
print("=== COSTE IA ===")
print(f"Coste IA trazado original: {coste_ia_trazado_usd_enero:,.2f} $")
resumen_importe("Coste IA trazado convertido a EUR", coste_ia_trazado_eur_enero)

print()
print("=== COSTES OPERATIVOS Y ESTRUCTURA ===")
resumen_importe("Coste plataforma usado en análisis", coste_plataforma_analisis)
resumen_importe("Comisiones Stripe", comisiones_stripe_estimadas)
resumen_importe("Costes fijos", costes_fijos_empresa)
resumen_importe("Marketing", marketing_enero)

print()
print("=== MÁRGENES ===")
resumen_importe("Margen frente a IA trazada convertida", margen_ia_trazado)
resumen_importe("Margen operativo plataforma", margen_operativo_plataforma)
resumen_importe("Resultado enero completo", resultado_enero_completo)

=== RESUMEN ECONÓMICO ENERO 2026 ===
Ingresos brutos Stripe: 3,942.99 €
Ingresos netos estimados sin IVA: 3,258.67 €

=== COSTE IA ===
Coste IA trazado original: 525.06 $
Coste IA trazado convertido a EUR: 447.05 €

=== COSTES OPERATIVOS Y ESTRUCTURA ===
Coste plataforma usado en análisis: 1,547.05 €
Comisiones Stripe: 209.25 €
Costes fijos: 6,000.00 €
Marketing: 12,000.00 €

=== MÁRGENES ===
Margen frente a IA trazada convertida: 2,811.62 €
Margen operativo plataforma: 1,502.37 €
Resultado enero completo: -16,497.63 €


In [124]:
resumen_economico_enero = pd.DataFrame([
    {"concepto": "Ingresos brutos Stripe", "importe": ingresos_brutos_enero, "tipo": "Ingreso"},
    {"concepto": "Ingresos netos estimados sin IVA", "importe": ingresos_netos_sin_iva_enero, "tipo": "Ingreso neto"},
    {"concepto": "Coste IA trazado convertido", "importe": coste_ia_trazado_eur_enero, "tipo": "Coste trazado"},
    {"concepto": "Coste plataforma usado en análisis", "importe": coste_plataforma_analisis, "tipo": "Coste operativo"},
    {"concepto": "Comisiones Stripe", "importe": comisiones_stripe_estimadas, "tipo": "Coste variable"},
    {"concepto": "Costes fijos", "importe": costes_fijos_empresa, "tipo": "Coste fijo"},
    {"concepto": "Marketing", "importe": marketing_enero, "tipo": "Marketing"},
    {"concepto": "Margen operativo plataforma", "importe": margen_operativo_plataforma, "tipo": "Margen"},
    {"concepto": "Resultado enero completo", "importe": resultado_enero_completo, "tipo": "Resultado"},
])

resumen_economico_enero

,concepto,importe,tipo
0,Ingresos brutos Stripe,"3,942.99",Ingreso
1,Ingresos netos estimados sin IVA,"3,258.67",Ingreso neto
2,Coste IA trazado convertido,447.05,Coste trazado
3,Coste plataforma usado en análisis,"1,547.05",Coste operativo
4,Comisiones Stripe,209.25,Coste variable
5,Costes fijos,"6,000.00",Coste fijo
6,Marketing,"12,000.00",Marketing
7,Margen operativo plataforma,"1,502.37",Margen
8,Resultado enero completo,"-16,497.63",Resultado


In [99]:
# Revisión de reembolsos
charges_enero[[
    "amount_eur",
    "amount_refunded",
    "refunded"
]].head()

print("Cobros enero:", len(charges_enero))
print("Reembolsados flag refunded:")
print(charges_enero["refunded"].value_counts(dropna=False))

charges_enero["amount_refunded_num"] = pd.to_numeric(
    charges_enero["amount_refunded"],
    errors="coerce"
).fillna(0)

print("Importe reembolsado enero, en céntimos:", charges_enero["amount_refunded_num"].sum())
print("Cargos con amount_refunded > 0:", (charges_enero["amount_refunded_num"] > 0).sum())

Cobros enero: 490
Reembolsados flag refunded:
refunded
False    490
Name: count, dtype: int64
Importe reembolsado enero, en céntimos: 0
Cargos con amount_refunded > 0: 0


In [100]:
# ============================================================
# Tabla de control del resultado económico completo
# ============================================================

control_resultado_enero = pd.DataFrame([
    {
        "concepto": "Ingresos netos estimados sin IVA",
        "signo": "+",
        "importe": ingresos_netos_sin_iva_enero,
        "fuente": "Calculado desde Stripe charges"
    },
    {
        "concepto": "Coste IA trazado",
        "signo": "-",
        "importe": coste_ia_analisis,
        "fuente": "Calculado desde ai_spans"
    },
    {
        "concepto": "Infraestructura, servidor, BBDD y compilaciones",
        "signo": "-",
        "importe": coste_infraestructura_empresa,
        "fuente": "Estimación a partir de información PROXUS"
    },
    {
        "concepto": "Comisiones Stripe",
        "signo": "-",
        "importe": comisiones_stripe_estimadas,
        "fuente": "Estimado a partir de tarifa Stripe"
    },
    {
        "concepto": "Costes fijos estructura",
        "signo": "-",
        "importe": costes_fijos_empresa,
        "fuente": "PROXUS"
    },
    {
        "concepto": "Marketing creadores TikTok",
        "signo": "-",
        "importe": marketing_enero,
        "fuente": "PROXUS"
    }
])

control_resultado_enero["importe_con_signo"] = np.where(
    control_resultado_enero["signo"] == "+",
    control_resultado_enero["importe"],
    -control_resultado_enero["importe"]
)

resultado_control = control_resultado_enero["importe_con_signo"].sum()

display(control_resultado_enero)

print(f"Resultado enero completo calculado por tabla de control: {resultado_control:,.2f} €")
print(f"Resultado enero completo anterior: {resultado_enero_completo:,.2f} €")
print(f"Diferencia: {resultado_control - resultado_enero_completo:,.6f} €")

,concepto,signo,importe,fuente,importe_con_signo
0,Ingresos netos estimados sin IVA,+,"3,258.67",Calculado desde Stripe charges,"3,258.67"
1,Coste IA trazado,-,447.05,Calculado desde ai_spans,-447.05
2,"Infraestructura, servidor, BBDD y compilaciones",-,"1,100.00",Estimación a partir de información PROXUS,"-1,100.00"
3,Comisiones Stripe,-,209.25,Estimado a partir de tarifa Stripe,-209.25
4,Costes fijos estructura,-,"6,000.00",PROXUS,"-6,000.00"
5,Marketing creadores TikTok,-,"12,000.00",PROXUS,"-12,000.00"


Resultado enero completo calculado por tabla de control: -16,497.63 €
Resultado enero completo anterior: -16,497.63 €
Diferencia: 0.000000 €


In [101]:
# ============================================================
# Validación de comisiones Stripe
# ============================================================

validacion_stripe = pd.DataFrame([
    {
        "concepto": "Ingresos brutos enero",
        "valor": ingresos_brutos_enero
    },
    {
        "concepto": "Número de cobros enero",
        "valor": n_cobros_enero
    },
    {
        "concepto": "Comisión porcentual estimada 2,2%",
        "valor": ingresos_brutos_enero * 0.022
    },
    {
        "concepto": "Comisión fija estimada 0,25€ por cobro",
        "valor": n_cobros_enero * 0.25
    },
    {
        "concepto": "Comisiones estimadas",
        "valor": comisiones_stripe_estimadas
    }
])

validacion_stripe

,concepto,valor
0,Ingresos brutos enero,"3,942.99"
1,Número de cobros enero,490.00
2,"Comisión porcentual estimada 2,2%",86.75
3,"Comisión fija estimada 0,25€ por cobro",122.50
4,Comisiones estimadas,209.25


In [102]:
# ============================================================
# KPIs básicos de monetización - Enero 2026
# ============================================================

arppu_bruto_enero = ingresos_brutos_enero / clientes_pago_enero
arppu_neto_enero = ingresos_netos_sin_iva_enero / clientes_pago_enero

coste_ia_trazado_eur_por_pagador = coste_ia_analisis / clientes_pago_enero

coste_plataforma_por_pagador = coste_plataforma_analisis / clientes_pago_enero
comision_stripe_por_pagador = comisiones_stripe_estimadas / clientes_pago_enero
marketing_por_pagador = marketing_enero / clientes_pago_enero
coste_fijo_por_pagador = costes_fijos_empresa / clientes_pago_enero

margen_ia_trazado_por_pagador = margen_ia_trazado / clientes_pago_enero
margen_operativo_por_pagador = margen_operativo_plataforma / clientes_pago_enero

kpis_monetizacion_enero = pd.DataFrame([
    {"KPI": "Clientes únicos con cobro", "valor": clientes_pago_enero, "unidad": "usuarios"},
    {"KPI": "ARPPU bruto", "valor": arppu_bruto_enero, "unidad": "€/pagador"},
    {"KPI": "ARPPU neto sin IVA", "valor": arppu_neto_enero, "unidad": "€/pagador"},

    {"KPI": "Coste IA trazado convertido por pagador", "valor": coste_ia_trazado_eur_por_pagador, "unidad": "€/pagador"},

    {"KPI": "Coste plataforma por pagador", "valor": coste_plataforma_por_pagador, "unidad": "€/pagador"},
    {"KPI": "Comisión Stripe por pagador", "valor": comision_stripe_por_pagador, "unidad": "€/pagador"},

    {"KPI": "Margen frente a IA trazada por pagador", "valor": margen_ia_trazado_por_pagador, "unidad": "€/pagador"},
    {"KPI": "Margen operativo plataforma por pagador", "valor": margen_operativo_por_pagador, "unidad": "€/pagador"},

    {"KPI": "Marketing por pagador", "valor": marketing_por_pagador, "unidad": "€/pagador"},
    {"KPI": "Coste fijo por pagador", "valor": coste_fijo_por_pagador, "unidad": "€/pagador"},
])

kpis_monetizacion_enero

,KPI,valor,unidad
0,Clientes únicos con cobro,489.00,usuarios
1,ARPPU bruto,8.06,€/pagador
2,ARPPU neto sin IVA,6.66,€/pagador
3,Coste IA trazado convertido por pagador,0.91,€/pagador
4,Coste plataforma por pagador,3.16,€/pagador
5,Comisión Stripe por pagador,0.43,€/pagador
6,Margen frente a IA trazada por pagador,5.75,€/pagador
7,Margen operativo plataforma por pagador,3.07,€/pagador
8,Marketing por pagador,24.54,€/pagador
9,Coste fijo por pagador,12.27,€/pagador


In [103]:
# ============================================================
# CAC aproximado - Enero 2026
# ============================================================

# CAC sobre clientes pagadores observados en enero
cac_pagador_enero = marketing_enero / clientes_pago_enero

# Nuevos usuarios registrados en enero
usuarios_enero = users[
    (users["created_at"] >= inicio_enero) &
    (users["created_at"] < fin_enero)
].copy()

n_usuarios_registrados_enero = usuarios_enero["id"].nunique()

cac_usuario_registrado_enero = marketing_enero / n_usuarios_registrados_enero

print("=== CAC APROXIMADO ENERO 2026 ===")
print(f"Nuevos usuarios registrados enero: {n_usuarios_registrados_enero:,}")
print(f"Clientes únicos con cobro enero: {clientes_pago_enero:,}")
print(f"CAC por usuario registrado: {cac_usuario_registrado_enero:,.2f} €")
print(f"CAC por cliente pagador: {cac_pagador_enero:,.2f} €")

=== CAC APROXIMADO ENERO 2026 ===
Nuevos usuarios registrados enero: 25,134
Clientes únicos con cobro enero: 489
CAC por usuario registrado: 0.48 €
CAC por cliente pagador: 24.54 €


## KPIs de conversión, retención y recurrencia

Una vez calculados los ingresos, costes y márgenes de enero, se estiman KPIs de negocio vinculados al modelo freemium: tasa de conversión, ARPU, ARPPU, CAC, MRR, churn observable y LTV aproximado. Estos indicadores permiten traducir los resultados analíticos a métricas de decisión empresarial.

In [104]:
# ============================================================
# KPIs de conversión
# ============================================================

usuarios_total = users["id"].nunique()
free_puros_total = users["es_free_puro"].sum()
pagadores_activos_total = users["es_pagador_activo"].sum()
churned_total = users["es_churned"].sum()
alguna_vez_pagaron_total = users["ha_pagado_alguna_vez"].sum()

tasa_conversion_acumulada = alguna_vez_pagaron_total / usuarios_total
tasa_conversion_activa = pagadores_activos_total / usuarios_total

usuarios_enero = users[
    (users["created_at"] >= inicio_enero) &
    (users["created_at"] < fin_enero)
].copy()

usuarios_enero_total = usuarios_enero["id"].nunique()
usuarios_enero_pagaron = usuarios_enero["ha_pagado_alguna_vez"].sum()
tasa_conversion_enero = usuarios_enero_pagaron / usuarios_enero_total

kpis_conversion = pd.DataFrame([
    {
        "KPI": "Tasa conversión acumulada",
        "numerador": alguna_vez_pagaron_total,
        "denominador": usuarios_total,
        "valor": tasa_conversion_acumulada,
        "valor_%": tasa_conversion_acumulada * 100,
        "formula": "Usuarios que alguna vez pagaron / usuarios totales",
        "interpretacion": "Conversión histórica acumulada del modelo freemium"
    },
    {
        "KPI": "Tasa conversión activa",
        "numerador": pagadores_activos_total,
        "denominador": usuarios_total,
        "valor": tasa_conversion_activa,
        "valor_%": tasa_conversion_activa * 100,
        "formula": "Pagadores activos / usuarios totales",
        "interpretacion": "Proporción de usuarios que pagan en la fecha de corte"
    },
    {
        "KPI": "Tasa conversión cohorte enero",
        "numerador": usuarios_enero_pagaron,
        "denominador": usuarios_enero_total,
        "valor": tasa_conversion_enero,
        "valor_%": tasa_conversion_enero * 100,
        "formula": "Usuarios registrados en enero que pagaron / usuarios registrados en enero",
        "interpretacion": "Conversión observada de la cohorte enero hasta la fecha de corte"
    }
])

kpis_conversion["valor_%"] = kpis_conversion["valor_%"].round(2)

kpis_conversion

,KPI,numerador,denominador,valor,valor_%,formula,interpretacion
0,Tasa conversión acumulada,619,32636,0.02,1.90,Usuarios que alguna vez pagaron / usuarios tot...,Conversión histórica acumulada del modelo free...
1,Tasa conversión activa,516,32636,0.02,1.58,Pagadores activos / usuarios totales,Proporción de usuarios que pagan en la fecha d...
2,Tasa conversión cohorte enero,475,25134,0.02,1.89,Usuarios registrados en enero que pagaron / us...,Conversión observada de la cohorte enero hasta...


In [105]:
# ============================================================
# KPI: MRR observado enero
# ============================================================

# En este análisis se aproxima el MRR a partir de los cobros recurrentes observados en enero.
# No se utilizan datos actuales proporcionados por PROXUS fuera del periodo analizado.

mrr_observado_enero_bruto = ingresos_brutos_enero
mrr_observado_enero_neto = ingresos_netos_sin_iva_enero

kpis_mrr = pd.DataFrame([
    {
        "KPI": "MRR observado enero bruto",
        "valor": mrr_observado_enero_bruto,
        "fuente": "Stripe charges enero",
        "formula": "Suma de cobros exitosos de enero",
        "interpretacion": "Aproximación al ingreso recurrente mensual observado en enero"
    },
    {
        "KPI": "MRR observado enero neto sin IVA",
        "valor": mrr_observado_enero_neto,
        "fuente": "Stripe charges enero / 1,21",
        "formula": "Cobros exitosos de enero sin IVA estimado",
        "interpretacion": "Aproximación neta al ingreso recurrente mensual observado"
    }
])

kpis_mrr

,KPI,valor,fuente,formula,interpretacion
0,MRR observado enero bruto,"3,942.99",Stripe charges enero,Suma de cobros exitosos de enero,Aproximación al ingreso recurrente mensual obs...
1,MRR observado enero neto sin IVA,"3,258.67","Stripe charges enero / 1,21",Cobros exitosos de enero sin IVA estimado,Aproximación neta al ingreso recurrente mensua...


El MRR se aproxima a partir de los cobros exitosos observados en Stripe durante enero de 2026. Esta aproximación no equivale necesariamente al MRR contable normalizado, ya que puede incluir altas, renovaciones, descuentos o prorrateos, pero permite estimar el ingreso recurrente mensual observado dentro del periodo analizado.

In [106]:
# ============================================================
# KPI: LTV y LTV/CAC estimado con datos observables
# ============================================================

# Churn observable a fecha de corte:
# usuarios que pagaron alguna vez pero ya no tienen suscripción activa
churn_pago_observable = churned_total / alguna_vez_pagaron_total

# Retención observable
retencion_pago_observable = pagadores_activos_total / alguna_vez_pagaron_total

# Margen operativo mensual por pagador
margen_operativo_por_pagador = margen_operativo_plataforma / clientes_pago_enero

# CAC calculado con datos de enero
cac_pagador_calculado = marketing_enero / clientes_pago_enero

# LTV aproximado
ltv_estimado = margen_operativo_por_pagador / churn_pago_observable

ltv_cac_estimado = ltv_estimado / cac_pagador_calculado

kpis_ltv = pd.DataFrame([
    {
        "KPI": "Churn pago observable",
        "valor": churn_pago_observable,
        "valor_%": churn_pago_observable * 100,
        "formula": "Churned / usuarios que alguna vez pagaron",
        "interpretacion": "Proporción observable de usuarios que pagaron y ya no están activos"
    },
    {
        "KPI": "Retención pago observable",
        "valor": retencion_pago_observable,
        "valor_%": retencion_pago_observable * 100,
        "formula": "Pagadores activos / usuarios que alguna vez pagaron",
        "interpretacion": "Proporción observable de usuarios que siguen activos"
    },
    {
        "KPI": "Margen operativo por pagador",
        "valor": margen_operativo_por_pagador,
        "valor_%": np.nan,
        "formula": "Margen operativo plataforma / clientes con cobro enero",
        "interpretacion": "Margen mensual directo por pagador"
    },
    {
        "KPI": "CAC por pagador",
        "valor": cac_pagador_calculado,
        "valor_%": np.nan,
        "formula": "Marketing enero / clientes con cobro enero",
        "interpretacion": "Coste medio de adquisición por cliente de pago"
    },
    {
        "KPI": "LTV estimado",
        "valor": ltv_estimado,
        "valor_%": np.nan,
        "formula": "Margen operativo por pagador / churn observable",
        "interpretacion": "Valor esperado aproximado por cliente bajo churn observable"
    },
    {
        "KPI": "LTV/CAC estimado",
        "valor": ltv_cac_estimado,
        "valor_%": np.nan,
        "formula": "LTV estimado / CAC por pagador",
        "interpretacion": "Relación entre valor esperado y coste de adquisición"
    }
])

kpis_ltv["valor_%"] = kpis_ltv["valor_%"].round(2)

kpis_ltv

,KPI,valor,valor_%,formula,interpretacion
0,Churn pago observable,0.17,16.64,Churned / usuarios que alguna vez pagaron,Proporción observable de usuarios que pagaron ...
1,Retención pago observable,0.83,83.36,Pagadores activos / usuarios que alguna vez pa...,Proporción observable de usuarios que siguen a...
2,Margen operativo por pagador,3.07,NaN,Margen operativo plataforma / clientes con cob...,Margen mensual directo por pagador
3,CAC por pagador,24.54,NaN,Marketing enero / clientes con cobro enero,Coste medio de adquisición por cliente de pago
4,LTV estimado,18.46,NaN,Margen operativo por pagador / churn observable,Valor esperado aproximado por cliente bajo chu...
5,LTV/CAC estimado,0.75,NaN,LTV estimado / CAC por pagador,Relación entre valor esperado y coste de adqui...


El LTV se estima de forma aproximada utilizando el margen operativo mensual por pagador y el churn observable en la base de datos. Esta métrica no debe interpretarse como LTV financiero definitivo, ya que el churn disponible no corresponde a una tasa mensual estabilizada, sino a la proporción de usuarios que han pagado alguna vez y ya no están activos en la fecha de corte.

In [107]:
# ============================================================
# Tabla maestra de KPIs BI calculados
# ============================================================

tabla_kpis_bi = pd.DataFrame([
    {
        "bloque": "Conversión",
        "KPI": "Tasa conversión acumulada",
        "valor": tasa_conversion_acumulada * 100,
        "unidad": "%",
        "formula": "Usuarios que alguna vez pagaron / usuarios totales",
        "interpretacion": "Conversión histórica acumulada del modelo freemium"
    },
    {
        "bloque": "Conversión",
        "KPI": "Tasa monetización activa",
        "valor": tasa_conversion_activa * 100,
        "unidad": "%",
        "formula": "Pagadores activos / usuarios totales",
        "interpretacion": "Proporción de usuarios activos que pagan en la fecha de corte"
    },
    {
        "bloque": "Conversión",
        "KPI": "Tasa conversión cohorte enero",
        "valor": tasa_conversion_enero * 100,
        "unidad": "%",
        "formula": "Usuarios registrados en enero que pagaron / usuarios registrados en enero",
        "interpretacion": "Conversión observada de la cohorte enero hasta la fecha de corte"
    },
    {
        "bloque": "Monetización",
        "KPI": "ARPPU neto enero",
        "valor": arppu_neto_enero,
        "unidad": "€/pagador",
        "formula": "Ingresos netos enero / clientes con cobro enero",
        "interpretacion": "Ingreso neto medio por cliente de pago"
    },
    {
        "bloque": "Adquisición",
        "KPI": "CAC por pagador",
        "valor": cac_pagador_calculado,
        "unidad": "€/pagador",
        "formula": "Marketing enero / clientes con cobro enero",
        "interpretacion": "Coste medio de adquisición por cliente de pago"
    },
    {
        "bloque": "Recurrencia",
        "KPI": "MRR observado enero bruto",
        "valor": mrr_observado_enero_bruto,
        "unidad": "€",
        "formula": "Cobros exitosos Stripe enero",
        "interpretacion": "Aproximación al ingreso recurrente mensual observado"
    },
    {
        "bloque": "Retención",
        "KPI": "Churn pago observable",
        "valor": churn_pago_observable * 100,
        "unidad": "%",
        "formula": "Churned / usuarios que alguna vez pagaron",
        "interpretacion": "Proporción observable de usuarios de pago que ya no están activos"
    },
    {
        "bloque": "Rentabilidad",
        "KPI": "Margen operativo plataforma",
        "valor": margen_operativo_plataforma,
        "unidad": "€",
        "formula": "Ingresos netos - costes plataforma - Stripe",
        "interpretacion": "Margen tras cubrir operación directa del producto"
    },
    {
        "bloque": "Rentabilidad",
        "KPI": "Resultado enero completo",
        "valor": resultado_enero_completo,
        "unidad": "€",
        "formula": "Ingresos netos - plataforma - Stripe - fijos - marketing",
        "interpretacion": "Resultado económico completo estimado del mes"
    },
    {
        "bloque": "Unit economics",
        "KPI": "LTV estimado",
        "valor": ltv_estimado,
        "unidad": "€/cliente",
        "formula": "Margen operativo por pagador / churn observable",
        "interpretacion": "Valor esperado aproximado del cliente"
    },
    {
        "bloque": "Unit economics",
        "KPI": "LTV/CAC estimado",
        "valor": ltv_cac_estimado,
        "unidad": "ratio",
        "formula": "LTV estimado / CAC por pagador",
        "interpretacion": "Relación entre valor esperado y coste de adquisición"
    }
])

tabla_kpis_bi.round(2)

,bloque,KPI,valor,unidad,formula,interpretacion
0,Conversión,Tasa conversión acumulada,1.90,%,Usuarios que alguna vez pagaron / usuarios tot...,Conversión histórica acumulada del modelo free...
1,Conversión,Tasa monetización activa,1.58,%,Pagadores activos / usuarios totales,Proporción de usuarios activos que pagan en la...
2,Conversión,Tasa conversión cohorte enero,1.89,%,Usuarios registrados en enero que pagaron / us...,Conversión observada de la cohorte enero hasta...
3,Monetización,ARPPU neto enero,6.66,€/pagador,Ingresos netos enero / clientes con cobro enero,Ingreso neto medio por cliente de pago
4,Adquisición,CAC por pagador,24.54,€/pagador,Marketing enero / clientes con cobro enero,Coste medio de adquisición por cliente de pago
5,Recurrencia,MRR observado enero bruto,"3,942.99",€,Cobros exitosos Stripe enero,Aproximación al ingreso recurrente mensual obs...
6,Retención,Churn pago observable,16.64,%,Churned / usuarios que alguna vez pagaron,Proporción observable de usuarios de pago que ...
7,Rentabilidad,Margen operativo plataforma,"1,502.37",€,Ingresos netos - costes plataforma - Stripe,Margen tras cubrir operación directa del producto
8,Rentabilidad,Resultado enero completo,"-16,497.63",€,Ingresos netos - plataforma - Stripe - fijos -...,Resultado económico completo estimado del mes
9,Unit economics,LTV estimado,18.46,€/cliente,Margen operativo por pagador / churn observable,Valor esperado aproximado del cliente


## Punto muerto económico

El punto muerto se calcula a partir de la fórmula clásica:

\[
Q^* = \frac{CF}{P - CV_u}
\]

donde \(Q^*\) representa el número de pagadores necesarios para cubrir los costes fijos, \(P\) es el ingreso neto medio por pagador y \(CV_u\) el coste variable unitario por pagador.

En este análisis se distingue entre dos escenarios:

1. **Punto muerto operativo sin marketing**, que mide cuántos pagadores serían necesarios para cubrir la estructura fija y la infraestructura recurrente.
2. **Punto muerto completo con marketing**, que incorpora además la inversión en adquisición de enero.

El coste variable unitario incluye costes directamente asociados al usuario de pago: IA y comisiones Stripe. La infraestructura se trata como coste fijo o semifijo de plataforma, ya que no depende linealmente de cada nuevo pagador en el corto plazo.

In [108]:
# ============================================================
# Punto muerto económico con CF y CVu
# ============================================================

# Ingreso neto medio por pagador
precio_medio_neto_pagador = arppu_neto_enero

# Coste variable total:
# IA calculada + comisiones Stripe estimadas
coste_variable_total = (
    coste_ia_analisis
    + comisiones_stripe_estimadas
)

# Coste variable unitario por pagador
coste_variable_unitario = coste_variable_total / clientes_pago_enero

# Margen de contribución unitario
margen_contribucion_unitario = (
    precio_medio_neto_pagador
    - coste_variable_unitario
)

# Costes fijos/semi-fijos
# Infraestructura se trata como coste fijo operativo de corto plazo
costes_fijos_operativos = (
    costes_fijos_empresa
    + coste_infraestructura_empresa
)

costes_fijos_completos_con_marketing = (
    costes_fijos_empresa
    + coste_infraestructura_empresa
    + marketing_enero
)

# Punto muerto
pm_pagadores_sin_marketing = (
    costes_fijos_operativos / margen_contribucion_unitario
    if margen_contribucion_unitario > 0
    else np.nan
)

pm_pagadores_con_marketing = (
    costes_fijos_completos_con_marketing / margen_contribucion_unitario
    if margen_contribucion_unitario > 0
    else np.nan
)

punto_muerto = pd.DataFrame([
    {
        "escenario": "Punto muerto operativo sin marketing",
        "costes_fijos": costes_fijos_operativos,
        "coste_variable_total": coste_variable_total,
        "coste_variable_unitario": coste_variable_unitario,
        "precio_medio_neto_pagador": precio_medio_neto_pagador,
        "margen_contribucion_unitario": margen_contribucion_unitario,
        "pagadores_necesarios": pm_pagadores_sin_marketing,
        "pagadores_observados_enero": clientes_pago_enero,
        "gap_pagadores": pm_pagadores_sin_marketing - clientes_pago_enero
    },
    {
        "escenario": "Punto muerto completo con marketing",
        "costes_fijos": costes_fijos_completos_con_marketing,
        "coste_variable_total": coste_variable_total,
        "coste_variable_unitario": coste_variable_unitario,
        "precio_medio_neto_pagador": precio_medio_neto_pagador,
        "margen_contribucion_unitario": margen_contribucion_unitario,
        "pagadores_necesarios": pm_pagadores_con_marketing,
        "pagadores_observados_enero": clientes_pago_enero,
        "gap_pagadores": pm_pagadores_con_marketing - clientes_pago_enero
    }
])

punto_muerto.round(2)

,escenario,costes_fijos,coste_variable_total,coste_variable_unitario,precio_medio_neto_pagador,margen_contribucion_unitario,pagadores_necesarios,pagadores_observados_enero,gap_pagadores
0,Punto muerto operativo sin marketing,"7,100.00",656.30,1.34,6.66,5.32,"1,334.13",489,845.13
1,Punto muerto completo con marketing,"19,100.00",656.30,1.34,6.66,5.32,"3,589.00",489,"3,100.00"


In [109]:
# ============================================================
# Resumen textual del punto muerto
# ============================================================

print("=== PUNTO MUERTO ECONÓMICO ===")
print(f"Pagadores observados en enero: {clientes_pago_enero:,.0f}")
print(f"Ingreso neto medio por pagador (P): {precio_medio_neto_pagador:,.2f} €")
print(f"Coste variable unitario (CVu): {coste_variable_unitario:,.2f} €")
print(f"Margen de contribución unitario (MCu): {margen_contribucion_unitario:,.2f} €")

print("\n--- Escenario 1: sin marketing ---")
print(f"Costes fijos operativos: {costes_fijos_operativos:,.2f} €")
print(f"Pagadores necesarios: {pm_pagadores_sin_marketing:,.0f}")
print(f"Gap frente a enero: {pm_pagadores_sin_marketing - clientes_pago_enero:,.0f}")

print("\n--- Escenario 2: con marketing ---")
print(f"Costes fijos completos con marketing: {costes_fijos_completos_con_marketing:,.2f} €")
print(f"Pagadores necesarios: {pm_pagadores_con_marketing:,.0f}")
print(f"Gap frente a enero: {pm_pagadores_con_marketing - clientes_pago_enero:,.0f}")

=== PUNTO MUERTO ECONÓMICO ===
Pagadores observados en enero: 489
Ingreso neto medio por pagador (P): 6.66 €
Coste variable unitario (CVu): 1.34 €
Margen de contribución unitario (MCu): 5.32 €

--- Escenario 1: sin marketing ---
Costes fijos operativos: 7,100.00 €
Pagadores necesarios: 1,334
Gap frente a enero: 845

--- Escenario 2: con marketing ---
Costes fijos completos con marketing: 19,100.00 €
Pagadores necesarios: 3,589
Gap frente a enero: 3,100


## KPIs de activación y eficiencia del plan gratuito

Además de los KPIs financieros, se analizan indicadores de uso para evaluar si los usuarios gratuitos experimentan valor, qué funcionalidades activan y si existe concentración de costes en una minoría intensiva de usuarios.

In [110]:
# ============================================================
# Cargar tablas de uso para KPIs de activación y eficiencia
# ============================================================

tests = pd.read_csv(DATA_PATH / "tests.csv", low_memory=False)
questions = pd.read_csv(DATA_PATH / "questions.csv", low_memory=False)

try:
    notes = pd.read_csv(DATA_PATH / "notes.csv", low_memory=False)
    cards = pd.read_csv(DATA_PATH / "cards.csv", low_memory=False)
    print("notes y cards cargadas")
except FileNotFoundError:
    notes = None
    cards = None
    print("notes/cards no encontradas")

try:
    chat_conversation = pd.read_csv(DATA_PATH / "chat_conversation.csv", low_memory=False)
    chat_messages = pd.read_csv(DATA_PATH / "chat_messages.csv", low_memory=False)
    chat_messages_chat_relation = pd.read_csv(DATA_PATH / "chat_messages_chat_relation.csv", low_memory=False)
    print("tablas de chat cargadas")
except FileNotFoundError:
    chat_conversation = None
    chat_messages = None
    chat_messages_chat_relation = None
    print("tablas de chat no encontradas")

print("tests:", tests.shape)
print("questions:", questions.shape)

notes y cards cargadas
tablas de chat cargadas
tests: (88995, 12)
questions: (1888944, 13)


In [111]:
# ============================================================
# Usuarios activados por funcionalidad
# ============================================================

# Tests / preguntas IA
tests_base = tests.copy()
tests_base["user_id"] = clean_id(tests_base["user_id"])
tests_base["created_at"] = parse_dates(tests_base["created_at"])

usuarios_tests = set(
    tests_base["user_id"].dropna().astype(str)
)

# Preguntas IA
questions_base = questions.copy()
questions_base["test_id"] = clean_id(questions_base["test_id"])

if "ai_assisted" in questions_base.columns:
    questions_base["ai_assisted_bool"] = (
        questions_base["ai_assisted"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "t", "yes"])
    )
else:
    questions_base["ai_assisted_bool"] = False

questions_ai = questions_base[questions_base["ai_assisted_bool"]].copy()

questions_ai_users = questions_ai.merge(
    tests_base[["id", "user_id"]].assign(id=lambda df: clean_id(df["id"])).rename(columns={"id": "test_id"}),
    on="test_id",
    how="left"
)

usuarios_preguntas_ia = set(
    questions_ai_users["user_id"].dropna().astype(str)
)

# Flashcards
usuarios_flashcards = set()

if notes is not None:
    notes_base = notes.copy()
    notes_base["user_id"] = clean_id(notes_base["user_id"])
    notes_base["created_at"] = parse_dates(notes_base["created_at"])

    usuarios_flashcards = set(
        notes_base["user_id"].dropna().astype(str)
    )

# Chat
usuarios_chat = set()

if chat_conversation is not None:
    chat_conv_base = chat_conversation.copy()
    chat_conv_base["user_id"] = clean_id(chat_conv_base["user_id"])

    usuarios_chat = set(
        chat_conv_base["user_id"].dropna().astype(str)
    )

usuarios_activados = usuarios_tests | usuarios_flashcards | usuarios_chat
usuarios_activados_ia = usuarios_preguntas_ia | usuarios_flashcards | usuarios_chat

print("Usuarios con tests:", len(usuarios_tests))
print("Usuarios con preguntas IA:", len(usuarios_preguntas_ia))
print("Usuarios con flashcards/notas:", len(usuarios_flashcards))
print("Usuarios con chat:", len(usuarios_chat))
print("Usuarios activados básicos:", len(usuarios_activados))
print("Usuarios activados IA:", len(usuarios_activados_ia))

Usuarios con tests: 18112
Usuarios con preguntas IA: 17611
Usuarios con flashcards/notas: 5932
Usuarios con chat: 2672
Usuarios activados básicos: 20581
Usuarios activados IA: 20122


In [112]:
# ============================================================
# KPIs de activación
# ============================================================

tasa_activacion_basica = len(usuarios_activados) / usuarios_total
tasa_activacion_ia = len(usuarios_activados_ia) / usuarios_total

tasa_uso_tests = len(usuarios_tests) / usuarios_total
tasa_uso_preguntas_ia = len(usuarios_preguntas_ia) / usuarios_total
tasa_uso_flashcards = len(usuarios_flashcards) / usuarios_total
tasa_uso_chat = len(usuarios_chat) / usuarios_total

kpis_activacion = pd.DataFrame([
    {
        "KPI": "Tasa activación básica",
        "valor_%": tasa_activacion_basica * 100,
        "formula": "Usuarios con al menos una acción de valor / usuarios totales",
        "interpretacion": "Usuarios que han experimentado valor inicial en la plataforma"
    },
    {
        "KPI": "Tasa activación IA",
        "valor_%": tasa_activacion_ia * 100,
        "formula": "Usuarios con uso de preguntas IA, flashcards o chat / usuarios totales",
        "interpretacion": "Usuarios que han utilizado alguna funcionalidad basada en IA"
    },
    {
        "KPI": "Uso de tests",
        "valor_%": tasa_uso_tests * 100,
        "formula": "Usuarios con tests / usuarios totales",
        "interpretacion": "Adopción de la funcionalidad central de estudio"
    },
    {
        "KPI": "Uso de preguntas IA",
        "valor_%": tasa_uso_preguntas_ia * 100,
        "formula": "Usuarios con preguntas IA / usuarios totales",
        "interpretacion": "Adopción de generación IA de preguntas"
    },
    {
        "KPI": "Uso de flashcards",
        "valor_%": tasa_uso_flashcards * 100,
        "formula": "Usuarios con flashcards / usuarios totales",
        "interpretacion": "Adopción de estudio mediante flashcards"
    },
    {
        "KPI": "Uso de chat",
        "valor_%": tasa_uso_chat * 100,
        "formula": "Usuarios con chat / usuarios totales",
        "interpretacion": "Adopción del asistente conversacional"
    }
])

kpis_activacion["valor_%"] = kpis_activacion["valor_%"].round(2)

kpis_activacion

,KPI,valor_%,formula,interpretacion
0,Tasa activación básica,63.06,Usuarios con al menos una acción de valor / us...,Usuarios que han experimentado valor inicial e...
1,Tasa activación IA,61.66,"Usuarios con uso de preguntas IA, flashcards o...",Usuarios que han utilizado alguna funcionalida...
2,Uso de tests,55.50,Usuarios con tests / usuarios totales,Adopción de la funcionalidad central de estudio
3,Uso de preguntas IA,53.96,Usuarios con preguntas IA / usuarios totales,Adopción de generación IA de preguntas
4,Uso de flashcards,18.18,Usuarios con flashcards / usuarios totales,Adopción de estudio mediante flashcards
5,Uso de chat,8.19,Usuarios con chat / usuarios totales,Adopción del asistente conversacional


In [113]:
# ============================================================
# Activación de usuarios registrados en enero
# ============================================================

usuarios_enero_ids = set(usuarios_enero["id"].astype(str))

usuarios_enero_activados = usuarios_enero_ids & usuarios_activados
usuarios_enero_activados_ia = usuarios_enero_ids & usuarios_activados_ia

tasa_activacion_enero = len(usuarios_enero_activados) / len(usuarios_enero_ids)
tasa_activacion_ia_enero = len(usuarios_enero_activados_ia) / len(usuarios_enero_ids)

kpis_activacion_enero = pd.DataFrame([
    {
        "KPI": "Activación básica cohorte enero",
        "usuarios": len(usuarios_enero_activados),
        "base": len(usuarios_enero_ids),
        "valor_%": tasa_activacion_enero * 100,
        "interpretacion": "Usuarios registrados en enero que realizaron al menos una acción de valor"
    },
    {
        "KPI": "Activación IA cohorte enero",
        "usuarios": len(usuarios_enero_activados_ia),
        "base": len(usuarios_enero_ids),
        "valor_%": tasa_activacion_ia_enero * 100,
        "interpretacion": "Usuarios registrados en enero que usaron alguna funcionalidad IA"
    }
])

kpis_activacion_enero["valor_%"] = kpis_activacion_enero["valor_%"].round(2)

kpis_activacion_enero

,KPI,usuarios,base,valor_%,interpretacion
0,Activación básica cohorte enero,17474,25134,69.52,Usuarios registrados en enero que realizaron a...
1,Activación IA cohorte enero,17286,25134,68.78,Usuarios registrados en enero que usaron algun...


In [114]:
# ============================================================
# Coste IA trazado por usuario y segmento
# Cruce correcto: ai_spans.trace_id -> traces.id
# ============================================================

traces_cost = traces.copy()
spans_cost = ai_spans.copy()

# Normalizar IDs
traces_cost["id"] = clean_id(traces_cost["id"])
traces_cost["user_id"] = clean_id(traces_cost["user_id"])

spans_cost["trace_id"] = clean_id(spans_cost["trace_id"])

# Fechas
if "created_at" in spans_cost.columns:
    spans_cost["created_at"] = parse_dates(spans_cost["created_at"])
elif "start_time" in spans_cost.columns:
    spans_cost["created_at"] = parse_dates(spans_cost["start_time"])
else:
    raise ValueError("No encuentro columna de fecha en ai_spans")

# Coste IA
spans_cost["cost_usd"] = pd.to_numeric(spans_cost["cost"], errors="coerce").fillna(0)
spans_cost["cost_eur"] = spans_cost["cost_usd"] * USD_EUR_ENERO_2026

# Merge correcto: trace_id de ai_spans con id de traces
spans_user = spans_cost.merge(
    traces_cost[["id", "user_id"]].rename(columns={"id": "trace_id"}),
    on="trace_id",
    how="left"
)

print("Spans totales:", len(spans_cost))
print("Spans con user_id tras merge:", spans_user["user_id"].notna().sum())
print("Spans sin user_id tras merge:", spans_user["user_id"].isna().sum())

# Nos quedamos con spans asignables a usuario
spans_user = spans_user.dropna(subset=["user_id"]).copy()

# Coste por usuario
coste_ia_usuario = (
    spans_user
    .groupby("user_id")
    .agg(
        coste_ia_eur=("cost_eur", "sum"),
        coste_ia_usd=("cost_usd", "sum"),
        n_spans=("trace_id", "count")
    )
    .reset_index()
)

# Añadir segmento de usuario
users_aux = users[[
    "id",
    "es_free_puro",
    "es_pagador_activo",
    "es_churned"
]].copy()

users_aux["user_id"] = clean_id(users_aux["id"])
coste_ia_usuario["user_id"] = clean_id(coste_ia_usuario["user_id"])

coste_ia_usuario = coste_ia_usuario.merge(
    users_aux[[
        "user_id",
        "es_free_puro",
        "es_pagador_activo",
        "es_churned"
    ]],
    on="user_id",
    how="left"
)

coste_ia_usuario["segmento"] = np.select(
    [
        coste_ia_usuario["es_free_puro"] == True,
        coste_ia_usuario["es_pagador_activo"] == True,
        coste_ia_usuario["es_churned"] == True
    ],
    [
        "free_puro",
        "pagador_activo",
        "churned"
    ],
    default="otro"
)

print("\nUsuarios con coste IA trazado:", coste_ia_usuario["user_id"].nunique())
print("Coste IA total reconstruido en USD:", coste_ia_usuario["coste_ia_usd"].sum())
print("Coste IA total reconstruido en EUR:", coste_ia_usuario["coste_ia_eur"].sum())

coste_ia_usuario.head()

Spans totales: 269383
Spans con user_id tras merge: 269383
Spans sin user_id tras merge: 0

Usuarios con coste IA trazado: 20514
Coste IA total reconstruido en USD: 578.774732
Coste IA total reconstruido en EUR: 492.78906394041996


,user_id,coste_ia_eur,coste_ia_usd,n_spans,es_free_puro,es_pagador_activo,es_churned,segmento
0,010836c7-d25d-76de-8a17-02a187087283,0.04,0.05,16,True,False,False,free_puro
1,0199e234-756a-7f40-90de-4691e1c6178b,0.10,0.11,495,False,False,True,churned
2,0199e73a-7d69-76dd-a457-e0bf3f438b48,0.11,0.13,32,True,False,False,free_puro
3,0199e796-bc63-74df-b7d5-4fc412ec08dc,0.00,0.00,2,True,False,False,free_puro
4,0199e7b9-8cfc-7f4a-9639-8953e0cf19e1,0.01,0.01,5,False,False,True,churned


In [115]:
# ============================================================
# Coste IA por segmento
# ============================================================

coste_segmento = (
    coste_ia_usuario
    .groupby("segmento")
    .agg(
        usuarios_con_coste=("user_id", "nunique"),
        coste_ia_total_eur=("coste_ia_eur", "sum"),
        coste_ia_total_usd=("coste_ia_usd", "sum"),
        coste_ia_medio_usuario_con_coste=("coste_ia_eur", "mean"),
        coste_ia_mediano_usuario_con_coste=("coste_ia_eur", "median"),
        spans_totales=("n_spans", "sum")
    )
    .reset_index()
)

bases_segmento = pd.DataFrame([
    {"segmento": "free_puro", "usuarios_segmento": free_puros_total},
    {"segmento": "pagador_activo", "usuarios_segmento": pagadores_activos_total},
    {"segmento": "churned", "usuarios_segmento": churned_total}
])

coste_segmento = coste_segmento.merge(
    bases_segmento,
    on="segmento",
    how="left"
)

coste_segmento["coste_ia_medio_sobre_base_total"] = (
    coste_segmento["coste_ia_total_eur"] /
    coste_segmento["usuarios_segmento"]
)

coste_segmento["pct_usuarios_segmento_con_coste"] = (
    coste_segmento["usuarios_con_coste"] /
    coste_segmento["usuarios_segmento"] * 100
)

coste_segmento

,segmento,usuarios_con_coste,coste_ia_total_eur,coste_ia_total_usd,coste_ia_medio_usuario_con_coste,coste_ia_mediano_usuario_con_coste,spans_totales,usuarios_segmento,coste_ia_medio_sobre_base_total,pct_usuarios_segmento_con_coste
0,churned,101,24.96,29.32,0.25,0.11,11462,103.00,0.24,98.06
1,free_puro,19568,380.72,447.15,0.02,0.01,208674,"32,017.00",0.01,61.12
2,otro,351,4.21,4.94,0.01,0.01,2421,NaN,NaN,NaN
3,pagador_activo,494,82.90,97.36,0.17,0.11,46826,516.00,0.16,95.74


In [116]:
# ============================================================
# Free loaders: concentración del coste IA en usuarios free
# ============================================================

coste_free = coste_ia_usuario[
    coste_ia_usuario["segmento"] == "free_puro"
].copy()

# Nos quedamos solo con usuarios con coste positivo
coste_free = coste_free[coste_free["coste_ia_eur"] > 0].copy()

# Ordenar de mayor a menor coste y resetear índice
coste_free = (
    coste_free
    .sort_values("coste_ia_eur", ascending=False)
    .reset_index(drop=True)
)

coste_free["coste_acumulado"] = coste_free["coste_ia_eur"].cumsum()
coste_free["pct_coste_acumulado"] = (
    coste_free["coste_acumulado"] /
    coste_free["coste_ia_eur"].sum()
)

# Primera posición donde se alcanza el 80% del coste
posicion_80 = coste_free[
    coste_free["pct_coste_acumulado"] >= 0.80
].index[0] + 1

usuarios_80_coste_free = posicion_80
pct_usuarios_80_coste_free = usuarios_80_coste_free / len(coste_free) * 100

coste_free_total = coste_free["coste_ia_eur"].sum()

print("=== CONCENTRACIÓN DEL COSTE IA EN FREE PUROS ===")
print(f"Usuarios free con coste IA trazado positivo: {len(coste_free):,}")
print(f"Coste IA free total trazado: {coste_free_total:,.2f} €")
print(f"Usuarios free necesarios para concentrar el 80% del coste: {usuarios_80_coste_free:,}")
print(f"% de usuarios free con coste que concentran el 80%: {pct_usuarios_80_coste_free:.2f}%")

=== CONCENTRACIÓN DEL COSTE IA EN FREE PUROS ===
Usuarios free con coste IA trazado positivo: 19,466
Coste IA free total trazado: 380.72 €
Usuarios free necesarios para concentrar el 80% del coste: 7,231
% de usuarios free con coste que concentran el 80%: 37.15%


In [117]:
kpis_free_loaders = pd.DataFrame([
    {
        "KPI": "Usuarios free con coste IA trazado",
        "valor": len(coste_free),
        "unidad": "usuarios",
        "interpretacion": "Usuarios gratuitos con consumo IA observable"
    },
    {
        "KPI": "Coste IA free total trazado",
        "valor": coste_free_total,
        "unidad": "€",
        "interpretacion": "Coste IA trazado asociado a usuarios free puros"
    },
    {
        "KPI": "Usuarios que concentran 80% del coste free",
        "valor": usuarios_80_coste_free,
        "unidad": "usuarios",
        "interpretacion": "Número de usuarios free intensivos que explican la mayor parte del coste"
    },
    {
        "KPI": "% usuarios free con coste que concentran 80%",
        "valor": pct_usuarios_80_coste_free,
        "unidad": "%",
        "interpretacion": "Grado de concentración del coste IA gratuito"
    }
])

kpis_free_loaders

,KPI,valor,unidad,interpretacion
0,Usuarios free con coste IA trazado,"19,466.00",usuarios,Usuarios gratuitos con consumo IA observable
1,Coste IA free total trazado,380.72,€,Coste IA trazado asociado a usuarios free puros
2,Usuarios que concentran 80% del coste free,"7,231.00",usuarios,Número de usuarios free intensivos que explica...
3,% usuarios free con coste que concentran 80%,37.15,%,Grado de concentración del coste IA gratuito


In [118]:
# ============================================================
# Margen esperado por usuario free
# ============================================================

coste_free_total_trazado = coste_segmento.loc[
    coste_segmento["segmento"] == "free_puro",
    "coste_ia_total_eur"
].iloc[0]

coste_free_medio_base_total = coste_free_total_trazado / free_puros_total

margen_esperado_free = (
    tasa_conversion_acumulada * ltv_estimado
    - coste_free_medio_base_total
)

kpi_margen_free = pd.DataFrame([
    {
        "KPI": "Coste IA medio por free puro",
        "valor": coste_free_medio_base_total,
        "unidad": "€/usuario free",
        "formula": "Coste IA free trazado / free puros",
        "interpretacion": "Coste IA medio diluido de mantener usuarios gratuitos"
    },
    {
        "KPI": "Valor esperado por usuario free",
        "valor": tasa_conversion_acumulada * ltv_estimado,
        "unidad": "€/usuario free",
        "formula": "Tasa conversión acumulada × LTV estimado",
        "interpretacion": "Valor esperado generado por conversión futura"
    },
    {
        "KPI": "Margen esperado por usuario free",
        "valor": margen_esperado_free,
        "unidad": "€/usuario free",
        "formula": "Tasa conversión × LTV - coste free medio",
        "interpretacion": "Aproximación a la sostenibilidad del usuario gratuito medio"
    }
])

kpi_margen_free

,KPI,valor,unidad,formula,interpretacion
0,Coste IA medio por free puro,0.01,€/usuario free,Coste IA free trazado / free puros,Coste IA medio diluido de mantener usuarios gr...
1,Valor esperado por usuario free,0.35,€/usuario free,Tasa conversión acumulada × LTV estimado,Valor esperado generado por conversión futura
2,Margen esperado por usuario free,0.34,€/usuario free,Tasa conversión × LTV - coste free medio,Aproximación a la sostenibilidad del usuario g...


In [119]:
pct_usuarios_80_coste_free_sobre_total_free = (
    usuarios_80_coste_free / free_puros_total * 100
)

print(f"% sobre todos los free puros: {pct_usuarios_80_coste_free_sobre_total_free:.2f}%")

% sobre todos los free puros: 22.58%


In [120]:
# ============================================================
# Ampliar tabla maestra de KPIs con activación y eficiencia free
# ============================================================

nuevos_kpis_bi = pd.DataFrame([
    {
        "bloque": "Activación",
        "KPI": "Tasa activación básica",
        "valor": tasa_activacion_basica * 100,
        "unidad": "%",
        "formula": "Usuarios con acción de valor / usuarios totales",
        "interpretacion": "Usuarios que experimentan valor inicial"
    },
    {
        "bloque": "Activación",
        "KPI": "Tasa activación IA",
        "valor": tasa_activacion_ia * 100,
        "unidad": "%",
        "formula": "Usuarios con uso IA / usuarios totales",
        "interpretacion": "Adopción de funcionalidades IA"
    },
    {
        "bloque": "Eficiencia free",
        "KPI": "Coste IA medio por free puro",
        "valor": coste_free_medio_base_total,
        "unidad": "€/free",
        "formula": "Coste IA free trazado / free puros",
        "interpretacion": "Coste IA medio diluido del segmento gratuito"
    },
    {
        "bloque": "Eficiencia free",
        "KPI": "% free que concentran 80% coste IA",
        "valor": pct_usuarios_80_coste_free,
        "unidad": "%",
        "formula": "Usuarios free necesarios para 80% coste / free con coste",
        "interpretacion": "Concentración de consumo gratuito"
    },
    {
        "bloque": "Eficiencia free",
        "KPI": "Margen esperado por usuario free",
        "valor": margen_esperado_free,
        "unidad": "€/free",
        "formula": "Tasa conversión × LTV - coste IA medio free",
        "interpretacion": "Sostenibilidad esperada del usuario gratuito medio"
    }
])

tabla_kpis_bi_ampliada = pd.concat(
    [tabla_kpis_bi, nuevos_kpis_bi],
    ignore_index=True
)

def formatear_kpi(row):
    valor = row["valor"]
    unidad = row["unidad"]

    if unidad == "%":
        return f"{valor:.2f}%"
    elif unidad in ["€/free", "€/pagador", "€/cliente"]:
        return f"{valor:.4f} €" if abs(valor) < 1 else f"{valor:.2f} €"
    elif unidad == "€":
        return f"{valor:,.2f} €"
    elif unidad == "ratio":
        return f"{valor:.2f}"
    else:
        return f"{valor:.4f}"

tabla_kpis_bi_ampliada["valor_formateado"] = tabla_kpis_bi_ampliada.apply(
    formatear_kpi,
    axis=1
)

tabla_kpis_bi_ampliada[
    ["bloque", "KPI", "valor_formateado", "unidad", "formula", "interpretacion"]
]

,bloque,KPI,valor_formateado,unidad,formula,interpretacion
0,Conversión,Tasa conversión acumulada,1.90%,%,Usuarios que alguna vez pagaron / usuarios tot...,Conversión histórica acumulada del modelo free...
1,Conversión,Tasa monetización activa,1.58%,%,Pagadores activos / usuarios totales,Proporción de usuarios activos que pagan en la...
2,Conversión,Tasa conversión cohorte enero,1.89%,%,Usuarios registrados en enero que pagaron / us...,Conversión observada de la cohorte enero hasta...
3,Monetización,ARPPU neto enero,6.66 €,€/pagador,Ingresos netos enero / clientes con cobro enero,Ingreso neto medio por cliente de pago
4,Adquisición,CAC por pagador,24.54 €,€/pagador,Marketing enero / clientes con cobro enero,Coste medio de adquisición por cliente de pago
5,Recurrencia,MRR observado enero bruto,"3,942.99 €",€,Cobros exitosos Stripe enero,Aproximación al ingreso recurrente mensual obs...
6,Retención,Churn pago observable,16.64%,%,Churned / usuarios que alguna vez pagaron,Proporción observable de usuarios de pago que ...
7,Rentabilidad,Margen operativo plataforma,"1,502.37 €",€,Ingresos netos - costes plataforma - Stripe,Margen tras cubrir operación directa del producto
8,Rentabilidad,Resultado enero completo,"-16,497.63 €",€,Ingresos netos - plataforma - Stripe - fijos -...,Resultado económico completo estimado del mes
9,Unit economics,LTV estimado,18.46 €,€/cliente,Margen operativo por pagador / churn observable,Valor esperado aproximado del cliente


El análisis de concentración del coste IA por usuario se calcula sobre todo el periodo con trazabilidad disponible en `ai_spans`, mientras que el análisis económico de rentabilidad se restringe a enero de 2026. Por ello, el coste IA total utilizado en la concentración por usuarios no coincide exactamente con el coste IA mensual de enero.

In [121]:
nuevos_kpis_extra = pd.DataFrame([
    {
        "bloque": "Eficiencia free",
        "KPI": "% total free que concentra 80% coste IA",
        "valor": pct_usuarios_80_coste_free_sobre_total_free,
        "unidad": "%",
        "formula": "Usuarios free necesarios para 80% coste / free puros totales",
        "interpretacion": "Peso de los usuarios intensivos sobre toda la base gratuita"
    }
])

tabla_kpis_bi_ampliada = pd.concat(
    [tabla_kpis_bi_ampliada, nuevos_kpis_extra],
    ignore_index=True
)
tabla_kpis_bi_ampliada["valor_formateado"] = tabla_kpis_bi_ampliada.apply(
    formatear_kpi,
    axis=1
)

tabla_kpis_bi_ampliada[
    ["bloque", "KPI", "valor_formateado", "unidad", "formula", "interpretacion"]
]

,bloque,KPI,valor_formateado,unidad,formula,interpretacion
0,Conversión,Tasa conversión acumulada,1.90%,%,Usuarios que alguna vez pagaron / usuarios tot...,Conversión histórica acumulada del modelo free...
1,Conversión,Tasa monetización activa,1.58%,%,Pagadores activos / usuarios totales,Proporción de usuarios activos que pagan en la...
2,Conversión,Tasa conversión cohorte enero,1.89%,%,Usuarios registrados en enero que pagaron / us...,Conversión observada de la cohorte enero hasta...
3,Monetización,ARPPU neto enero,6.66 €,€/pagador,Ingresos netos enero / clientes con cobro enero,Ingreso neto medio por cliente de pago
4,Adquisición,CAC por pagador,24.54 €,€/pagador,Marketing enero / clientes con cobro enero,Coste medio de adquisición por cliente de pago
5,Recurrencia,MRR observado enero bruto,"3,942.99 €",€,Cobros exitosos Stripe enero,Aproximación al ingreso recurrente mensual obs...
6,Retención,Churn pago observable,16.64%,%,Churned / usuarios que alguna vez pagaron,Proporción observable de usuarios de pago que ...
7,Rentabilidad,Margen operativo plataforma,"1,502.37 €",€,Ingresos netos - costes plataforma - Stripe,Margen tras cubrir operación directa del producto
8,Rentabilidad,Resultado enero completo,"-16,497.63 €",€,Ingresos netos - plataforma - Stripe - fijos -...,Resultado económico completo estimado del mes
9,Unit economics,LTV estimado,18.46 €,€/cliente,Margen operativo por pagador / churn observable,Valor esperado aproximado del cliente


## Resumen metodológico del notebook

**Objetivo del notebook:**  
Analizar la sostenibilidad económica del modelo freemium de PROXUS mediante ingresos, costes, márgenes y KPIs de negocio. El notebook construye una línea base económica de enero de 2026 que servirá como referencia para la simulación de escenarios de límites del Notebook 06.

**Población o muestra analizada:**  
El análisis parte de la base total de 32.636 usuarios activos a fecha de corte del 7 de febrero de 2026. Para el análisis económico mensual se utilizan los cobros exitosos de Stripe correspondientes a enero de 2026 y los costes de IA trazados durante ese mismo mes. Para el análisis de activación y concentración del coste IA se utiliza la base de usuarios con actividad observable en las tablas de uso y trazabilidad.

**Periodo temporal utilizado:**  
El análisis económico principal se centra en enero de 2026, al tratarse del mes completo más representativo del periodo analizado. Algunos KPIs, como conversión acumulada, monetización activa y churn observable, se calculan con la situación acumulada hasta la fecha de corte del 7 de febrero de 2026. El análisis de concentración del coste IA por usuario utiliza todo el periodo con trazabilidad disponible en `ai_spans`.

**Tablas de origen utilizadas:**  
`users`, `stripe_charges`, `ai_spans`, `traces` y tablas de uso necesarias para calcular activación (`tests`, `questions`, `notes`, `cards`, `chat_conversation`, `chat_messages` y `chat_messages_chat_relation`). Además, se incorporan datos agregados proporcionados por PROXUS para partidas no observables directamente: infraestructura, costes fijos y marketing.

**Variables y métricas generadas:**  
Ingresos brutos, ingresos netos sin IVA, coste IA trazado, comisiones Stripe estimadas, infraestructura, margen operativo, resultado completo, ARPPU, CAC, MRR observado, churn observable, LTV estimado, LTV/CAC, tasa de activación, coste IA medio por usuario free, concentración del coste IA y punto muerto económico.

**Resultados principales:**  
En enero de 2026, PROXUS generó 3.942,99 € de ingresos brutos, equivalentes a 3.258,67 € netos estimados sin IVA. El coste IA trazado del mes fue de 447,05 € tras conversión a euros, y el coste de infraestructura, servidor, base de datos y compilaciones se fija en 1.100 €. Con estas hipótesis, el margen operativo de plataforma fue positivo, 1.502,37 €, pero el resultado completo del mes fue negativo, -16.497,63 €, al incorporar costes fijos y marketing.

Los KPIs muestran una tasa de conversión acumulada del 1,90%, una tasa de monetización activa del 1,58%, un ARPPU neto de 6,66 € y un CAC por pagador de 24,54 €. El LTV estimado asciende a 18,46 €, generando un ratio LTV/CAC de 0,75. Por tanto, bajo las condiciones observadas en enero, el valor esperado del cliente no compensa todavía su coste de adquisición.

La activación de usuarios es elevada: el 63,06% de la base realiza alguna acción de valor y el 61,66% utiliza alguna funcionalidad IA. Sin embargo, esta activación no se traduce proporcionalmente en conversión. El coste IA medio por usuario free puro es reducido, 0,0119 €, y el 37,15% de los usuarios free con coste IA positivo concentra el 80% del coste IA gratuito. Esto indica que el coste medio del usuario gratuito no parece ser el principal problema económico del modelo.

El punto muerto económico muestra que PROXUS habría necesitado aproximadamente 1.334 pagadores para cubrir costes operativos sin marketing y 3.589 pagadores para cubrir también la inversión comercial de enero, frente a los 489 clientes con cobro observados.

**Figuras o tablas que alimentan el TFG:**  
Tabla de ingresos y costes de enero, tabla de márgenes, tabla maestra de KPIs, tabla de activación y eficiencia del plan gratuito, tabla de concentración del coste IA y tabla de punto muerto económico.

**Relación con otros notebooks:**  
El Notebook 05 traduce los análisis previos de usuarios, conversión, límites y modelos inferenciales en indicadores económicos. Sus resultados sirven como línea base para el Notebook 06, donde se simulará el efecto de modificar los límites del plan gratuito.

**Limitaciones metodológicas:**  
El análisis combina datos calculados directamente desde las tablas con información agregada proporcionada por PROXUS. Los ingresos netos se estiman asumiendo que los importes de Stripe incluyen IVA. El MRR se aproxima mediante cobros observados en enero y no equivale necesariamente a un MRR contable normalizado. El churn observable no representa una tasa mensual estabilizada. El LTV debe interpretarse como una estimación aproximada. El coste de infraestructura se incorpora como partida agregada no observable directamente en la base de datos.

**Conclusión operativa para el TFG:**  
El reto económico principal no parece estar en el coste variable medio de IA ni en el usuario gratuito medio, sino en mejorar la relación entre conversión, CAC, churn y LTV. La plataforma activa a una proporción elevada de usuarios, pero necesita convertir y retener mejor para absorber costes fijos, infraestructura y marketing.

**Clasificación de resultados:**  
- Mantener como definitivos: ingresos enero, coste IA calculado, margen operativo, resultado completo, KPIs BI, activación, eficiencia free y punto muerto.
- Usar con cautela: LTV, LTV/CAC, MRR observado y churn observable.
- Mover a otro apartado: simulaciones de límites al Notebook 06.
- Descartar: métricas actuales proporcionadas por PROXUS que no puedan verificarse con los datos analizados.